# 3. データを前処理する

ここまでで、データには2つの問題があると分かった。

- `age` と `embarked` に**欠損**がある
- `sex` と `embarked` が**文字列**で、そのままではモデルに渡せない

この章で両方を片付けて、全ての値が数値で欠損のない状態にする。

## 1. 学習用と評価用を結合する

前処理は train と test の**両方に同じ処理**をしなければならない。
train だけ欠損を埋めても、test に欠損が残っていては予測できないため。

2回同じコードを書くと片方だけ直し忘れる事故が起きるので、
いったん縦に繋げて1つの表として処理し、最後に分け直す。

In [ ]:
import pandas as pd

train = pd.read_csv("../data/raw/train.csv", index_col=0)
test = pd.read_csv("../data/raw/test.csv", index_col=0)

# 後で分け直すために、どの id が train でどの id が test だったかを控えておく
#   結合すると両者の区別が付かなくなるため
train_index = train.index
test_index = test.index

In [ ]:
# pd.concat() は複数の表を繋げる。既定では縦（行方向）に積む
data = pd.concat([train, test])

print("train:", train.shape, "+ test:", test.shape, "→ data:", data.shape)
print()
print(data.dtypes)

# 出力の見方
#   445 行 + 446 行 = 891 行。列は train に合わせて 8 列になる
#   test には survived が無いため、test 由来の 446 行の survived は欠損（NaN）で埋まる
#   その結果 survived の型が int64 から float64 に変わる。
#   pandas では欠損を含む整数列が自動的に小数として扱われるため（01 で見た age と同じ理由）

### 出力の最後に付く `dtype: object` とは

`data.dtypes` のような**1列だけの表**（Series）を表示すると、最後に必ず
`dtype: ...` という行が付く。これはデータの一部ではなく、
**今表示した Series 自身の型**を pandas が添えているもの。

```
survived    float64     ← 左が列名（キー）、右がその列の型（バリュー）
pclass        int64
sex             str
...
dtype: object           ← この Series が何を入れているか
```

`object` は「数値でも文字列でもない、その他なんでも入る箱」を指す。
ここでは中身が `float64` のような**型そのもの**なので、`object` になる。

この注記は Series を表示すれば毎回出る。

| 出力 | 最後の行 | 理由 |
| --- | --- | --- |
| `value_counts()` | `Name: count, dtype: int64` | 人数を数えたので整数 |
| `corrwith()` | `dtype: float64` | 相関係数は小数 |
| `dtypes` | `dtype: object` | 中身が型オブジェクト |

In [ ]:
# 結合後の欠損を数える
data.isna().sum()

# 出力の見方
#   survived  446 → test 由来。これは埋めるべき欠損ではなく、これから予測する対象そのもの
#   age       177 → train 85 + test 92。これが埋める対象
#   embarked    2 → train のみ。これも埋める対象

### `isna()` の読み方

`isna` は **is NA**、つまり「NA かどうか」を尋ねるメソッド。
**NA** は **Not Available**（値が手に入っていない）の略で、統計の世界の言い方。

pandas で「値がない」を表すものは1種類ではないが、`isna()` はまとめて `True` にする。

| 表示 | 正体 | `isna()` |
| --- | --- | --- |
| `NaN` | Not a Number。小数の世界の「数値でない」印 | True |
| `None` | Python 本来の「何もない」 | True |
| `<NA>` | pandas の欠損専用の印 | True |

CSV の空欄は、数値の列なら `NaN` として読み込まれる。

`isna()` が返すのは、元と**同じ形の True / False の表**。

```
      age  embarked
id
3   False     False
4   False     False
```

そこに `.sum()` を付けると、`True` を 1 として数えるので欠損の個数になる。
`.sum().sum()` まで付ければ、表全体の合計が1つの数字で出る。

逆を調べたいときは `notna()`（欠損でなければ True）を使う。
`isnull()` という別名もあるが、結果は `isna()` と同一なので、どちらかに統一すればよい。

## 2. 欠損を埋める

欠損の埋め方に唯一の正解はない。代表的な選択肢は次の3つ。

| 方法 | 向いている場面 | 今回 |
| --- | --- | --- |
| 行を捨てる | 欠損がごく少数のとき | `age` は 891 人中 177 人（約20%）。捨てると学習データが大幅に減るので不採用 |
| 列ごと捨てる | その列がほとんど欠損のとき | `age` は8割が埋まっており、捨てるには惜しい |
| 代表値で埋める | 上記以外 | **これを採用する** |

数値の列（`age`）は平均値、カテゴリの列（`embarked`）は最頻値で埋める。
カテゴリに平均は存在しないので、埋め方は列の種類で決まる。

In [ ]:
# .fillna(値) は欠損をその値で置き換える
#   .mean() は平均値
data["age"] = data["age"].fillna(data["age"].mean())

print("埋めた値（全体の平均年齢）:", round(data["age"].mean(), 2))
print("age の欠損:", data["age"].isna().sum())

# 平均か中央値かは悩みどころ。今回の age は平均 29.70 / 中央値 28.0 とほぼ差が無いので
# どちらでも大差ない。fare のように極端な外れ値がある列なら中央値の方が無難

In [ ]:
# .mode() は最頻値（最も多く出てくる値）
#   末尾の [0] が重要。理由は下で説明する
data["embarked"] = data["embarked"].fillna(data["embarked"].mode()[0])

print("埋めた値（最も多い乗船港）:", data["embarked"].mode()[0])
print("embarked の欠損:", data["embarked"].isna().sum())

### `.mode()` に `[0]` を付ける理由

`.mode()` が返すのは値そのものではなく、**値の入ったリスト**（Series）。
最頻値は同数で並ぶことがあるので、1つでも常にリストで返ってくる。
`[0]` はその1番目を取り出す指定。

付け忘れると `.fillna()` が「この値で埋めろ」ではなく
「行番号を突き合わせて埋めろ」と解釈し、**エラーも出ないまま1件も埋まらない**。
埋めた直後に欠損数を出しているのはこのため。

### 同率だったらどうするのか

最頻値が同数で並んだとき、`[0]` は「アルファベット順で最初のもの」を選ぶ。
恣意的に見えるが、**毎回同じ結果になる**のが利点。

ランダムに選ぶ手もあるが、採用しない。前処理に乱数が入ると実行のたびに
データが変わり、精度が上下したとき改良の成果か偶然かを区別できなくなる。
同率ということは「どちらも同じくらい普通」なので、選び方に悩む価値も薄い。

なお今回の `embarked` は `S` が 644 件、`C` が 168 件、`Q` が 77 件で同率ではない。
`[0]` が必要なのは、値が1つでも**戻り値の形がリストで固定されている**ため。

## 3. カテゴリ変数をダミー化する

文字列のままではモデルが扱えないので、02 で見た `get_dummies()` で 0/1 の列に分解する。

In [ ]:
data = pd.get_dummies(data)

print(data.shape)
print()
print(data.dtypes)

# 出力の見方
#   8 列 → 11 列に増えた
#     sex（1列）      → sex_female, sex_male（2列）
#     embarked（1列） → embarked_C, embarked_Q, embarked_S（3列）
#   文字列の列が無くなり、数値と bool だけになった

## 4. 学習用と評価用に分け直す

控えておいた id を使って元の2つに戻す。

In [ ]:
# .loc[id のリスト] で、指定した id の行だけを取り出す
train = data.loc[train_index]
test = data.loc[test_index]

# test の survived は結合時に付いた空の列なので落とす
#   columns= を使うと「列を落とす」ことが名前から分かる
#   行を落とすときは index= を使う
test = test.drop(columns=["survived"])

print("train:", train.shape)  # 11 列（survived を含む）
print("test:", test.shape)  # 10 列（survived を除いた分だけ少ない）

In [ ]:
train.head()

# 出力の見方
#   survived が 1.0 / 0.0 と小数なのは、結合したときに float64 になった名残
#   train 側は全員分の答えが揃っているので、欠損はない

In [ ]:
# 仕上げの確認。ここが全て 0 でなければ、モデルに渡す前に原因を潰す
print("train の欠損:", train.isna().sum().sum())
print("test の欠損:", test.isna().sum().sum())
print("文字列の列:", list(train.select_dtypes("str").columns))

## 5. 次の章で使えるように保存する

Notebook ごとにメモリは独立しているので、ここで作った `train` と `test` は
この Notebook を閉じると消える。ファイルに書き出しておく。

保存先の `data/processed/` は `.gitignore` で除外されているため、git には記録されない。
競技データを再配布しないための措置で、加工後のデータも同じ扱いになる。

In [ ]:
# index=True で id も一緒に書き出す。id を失うと提出時にどの乗客の予測か分からなくなる
train.to_csv("../data/processed/train_processed.csv", index=True)
test.to_csv("../data/processed/test_processed.csv", index=True)

print("保存した")

## この章のまとめ

- train と test を**結合してから**処理した。同じ処理を確実に両方へ適用するため
- `age` は平均値（29.70）、`embarked` は最頻値（`S`）で欠損を埋めた
- `.mode()` は Series を返すので `[0]` が要る。忘れると**エラーも出ずに欠損が残る**
- `get_dummies()` で文字列を 0/1 の列に分解し、8 列 → 11 列になった
- 結果を `data/processed/` の2ファイル に保存した

なお今回は train と test を結合した状態で平均値や最頻値を求めている。
手順は単純になるが、厳密には本来知らないはずの評価用データの分布を使って
学習用データを加工していることになる。train だけから求めた値で両方を埋める方が
安全なので、精度を詰める段階で見直す余地がある。

次はいよいよモデルを作って、生存を予測する。